In [58]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

In [59]:
def load(path):
    df = pd.json_normalize(pd.read_json(path, lines=True).to_dict('records'))
    return df.set_index('doc_id').dropna(axis=1, how='all')

test_predictions = load('data/test_predictions.jsonl')
train_labels = load('data/train_labels.jsonl')
train_predictions = load('data/train_predictions.jsonl')
FIELDS = ["Last_name_ru", "First_name_ru", "Middle_name_ru", "Sex_ru", "Birth_date",
          "Birth_place_ru", "Issue_date", "Issue_organisation_code", "Licence_number",
          "Issue_organization_ru"]
def norm(s):
    return " ".join(s.split()).upper() if isinstance(s, str) else ""

In [60]:
train_predictions.head(1)

,source,quality.sharpness,quality.doc_px,quality.glare,viz.Last_name_ru.box,viz.Last_name_ru.conf,viz.Last_name_ru.lines,viz.First_name_ru.box,viz.First_name_ru.conf,viz.First_name_ru.lines,...,viz.Issue_organisation_code.box,viz.Issue_organisation_code.conf,viz.Issue_organisation_code.lines,viz.Licence_number.box,viz.Licence_number.conf,viz.Licence_number.lines,viz.Issue_organization_ru.box,viz.Issue_organization_ru.conf,viz.Issue_organization_ru.lines,mrz.lines
doc_id,,,,,,,,,,,,,,,,,,,,,
tr_00001,web,198.5,1714,0.101,"[215, 40, 484, 32]",0.9996,"[{'text': 'ЗАИНТИНОВ', 'box': [213, 40, 462, 28], 'char_conf': ['0.9992', '1.0', '0.9992', '0.9997', '0.9999', '0.9999', '0.9998', '0.9999', '0.9985']}]","[185, 82, 516, 32]",0.9992,"[{'text': 'ЗАКЕРИЙ', 'box': [184, 86, 482, 29], 'char_conf': ['0.9998', '0.9997', '0.9976', '0.9993', '0.9998', '0.9986', '0.9998']}]",...,"[180, 361, 408, 32]",0.9993,"[{'text': '620-004', 'box': [183, 363, 401, 29], 'char_conf': ['0.9998', '0.9993', '1.0', '0.9965', '1.0', '0.9996', '0.9998']}]","[183, 403, 446, 30]",0.9994,"[{'text': '61 24 283968', 'box': [183, 405, 341, 28], 'char_conf': ['0.9997', '0.9996', '0.9996', '1.0', '0.9994', '0.9999', '0.9998', '0.9966', '0.9991', '1.0', '0.9991', '0.9997']}]","[182, 443, 513, 91]",0.9945,"[{'text': 'ОТДЕЛ ПО ВОПРОСАМ МИГРАЦИИ ОМВД', 'box': [184, 444, 487, 29], 'char_conf': ['0.9996', '0.9991', '0.9996', '0.9999', '0.9995', '0.9999', '1.0', '1.0', '0.9974', '1.0', '0.999', '1.0', '0.9997', '0.9994', '0.9999', '0.9998', '0.9989', '0.9998', '0.9999', '0.9997', '0.9994', '0.9993', '0.9998', '0.9995', '0.9998', '0.9995', '1.0', '0.9994', '0.9979', '0.9969', '0.9999']}, {'text': 'РОССИИ ПО РЯЗАНСКОЙ СБЛАСТИ В', 'box': [182, 473, 409, 30], 'char_conf': ['0.9979', '0.9997', '0.9999', '0.9995', '1.0', '0.9999', '0.9999', '0.9997', '0.999', '0.9983', '0.9999', '0.9998', '1.0', '1.0', '0.9999', '0.9987', '0.9978', '0.9987', '0.9989', '0.9952', '0.6197', '1.0', '0.9998', '0.9999', '0.9995', '0.9963', '0.9996', '0.9998', '0.9994']}, {'text': 'КАСИМОВСКОМ РАЙОНЕ', 'box': [180, 501, 491, 28], 'char_conf': ['1.0', '0.9991', '0.9996', '1.0', '0.9973', '0.9981', '0.9995', '0.9996', '0.9997', '0.9999', '0.9999', '1.0', '0.9998', '0.9998', '0.9998', '0.9997', '0.9997', '0.9989']}]","[{'text': 'PNRUSZAINTINOV<<ZAKERIQ<NAZIFULOVI3<<<<<<<<<', 'char_conf': ['0.9999', '0.9999', '0.9991', '0.9996', '0.9998', '0.9999', '0.9999', '0.9999', '0.9998', '1.0', '0.9997', '0.9963', '0.9991', '0.9999', '0.9991', '0.9997', '0.9998', '0.9996', '0.9973', '1.0', '0.9993', '0.9999', '0.9997', '0.9999', '1.0', '0.9999', '0.9998', '0.9999', '0.9999', '0.9998', '1.0', '0.9999', '0.9999', '0.9976', '0.9997', '1.0', '0.9979', '0.9998', '0.9998', '0.9995', '0.9999', '0.9993', '0.9998', '0.9999']}, {'text': '6122839687RUS1006102M<<<<<<<4250911620004<28', 'char_conf': ['1.0', '0.9999', '0.9995', '1.0', '0.9991', '0.9965', '0.9996', '0.9999', '0.9999', '0.9998', '0.9998', '0.9994', '0.9998', '0.9996', '0.9996', '0.9999', '0.9999', '0.9997', '0.9999', '0.9982', '1.0', '0.9997', '0.9999', '0.9998', '0.9981', '0.9992', '0.9986', '0.9987', '0.9992', '0.9999', '0.9994', '0.9988', '0.9994', '0.9985', '0.9973', '1.0', '0.9987', '0.9995', '0.9998', '0.9999', '0.9992', '1.0', '0.9986', '0.9996']}]"


In [61]:
train_labels.head(1)

,fields.Last_name_ru,fields.First_name_ru,fields.Middle_name_ru,fields.Sex_ru,fields.Birth_date,fields.Birth_place_ru,fields.Issue_date,fields.Issue_organisation_code,fields.Licence_number,fields.Issue_organization_ru
doc_id,,,,,,,,,,
tr_00001,ЗАИНТИНОВ,ЗАКЕРИЙ,НАЗИФУЛОВИЧ,МУЖ.,10.06.2010,С. ТУРУНОВО БАТЫРЕВСКОГО Р-НА,11.09.2025,620-004,61 24 283968,ОТДЕЛ ПО ВОПРОСАМ МИГРАЦИИ ОМВД РОССИИ ПО РЯЗАНСКОЙ ОБЛАСТИ В КАСИМОВСКОМ РАЙОНЕ


In [62]:
train_predictions.isna().sum().sum()

np.int64(1233)

In [63]:
pd.set_option('display.max_colwidth', None)
train_predictions.head(1).T

doc_id,tr_00001
source,web
quality.sharpness,198.5
quality.doc_px,1714
quality.glare,0.101
viz.Last_name_ru.box,"[215, 40, 484, 32]"
viz.Last_name_ru.conf,0.9996
viz.Last_name_ru.lines,"[{'text': 'ЗАИНТИНОВ', 'box': [213, 40, 462, 28], 'char_conf': ['0.9992', '1.0', '0.9992', '0.9997', '0.9999', '0.9999', '0.9998', '0.9999', '0.9985']}]"
viz.First_name_ru.box,"[185, 82, 516, 32]"
viz.First_name_ru.conf,0.9992
viz.First_name_ru.lines,"[{'text': 'ЗАКЕРИЙ', 'box': [184, 86, 482, 29], 'char_conf': ['0.9998', '0.9997', '0.9976', '0.9993', '0.9998', '0.9986', '0.9998']}]"


In [78]:
two_fields = ['Birth_place_ru', 'Issue_organization_ru']
name_fields = ['Last_name_ru', 'First_name_ru', 'Middle_name_ru']
mrz_fields = ['Birth_date', 'Issue_date', 'Issue_organisation_code', 'Licence_number']

def field_quality(box, lines):
    if not isinstance(lines, list):
        return np.nan, np.nan
    line_h = np.mean([l['box'][3] for l in lines])
    confs = []
    for l in lines:
        confs += l['char_conf']
    return min(confs), round(box[3] / line_h) - len(lines)

def text_conf(lines):
    if not isinstance(lines, list):
        return '', []
    conf = []
    for l in lines:
        conf += l['char_conf'] + [1.0]
    return ' '.join(l['text'] for l in lines), conf

def line_text(lines, i):
    if not isinstance(lines, list) or len(lines) <= i:
        return None
    return lines[i]['text']

def add_quality(df):
    for f in FIELDS:
        df[f'viz.{f}.chars'] = df[f'viz.{f}.lines'].map(text_conf)
        df[f'viz.{f}.text'] = df[f'viz.{f}.chars'].map(lambda chars: norm(chars[0]))
        for i, k in enumerate('xywh'):
            df[f'viz.{f}.box_{k}'] = df[f'viz.{f}.box'].map(lambda box: box[i] if isinstance(box, list) else np.nan)
    for f in two_fields:
        quality = [field_quality(box, lines) for box, lines in zip(df[f'viz.{f}.box'], df[f'viz.{f}.lines'])]
        df[f'viz.{f}.cmin'] = [q[0] for q in quality]
        df[f'viz.{f}.lost'] = [q[1] for q in quality]
    df['mrz.chars'] = df['mrz.lines'].map(text_conf)
    df['mrz.line0'] = df['mrz.lines'].map(lambda lines: line_text(lines, 0))
    df['mrz.line1'] = df['mrz.lines'].map(lambda lines: line_text(lines, 1))
    return df


train_predictions = add_quality(train_predictions)
test_predictions = add_quality(test_predictions)

парсим line2 mrz

In [79]:
def check(s, digit):
    s = s.replace('<', '0')
    if not s.isdigit():
        return False
    weights = [7, 3, 1]
    total = 0
    for i, c in enumerate(s):
        total += int(c) * weights[i % 3]
    return str(total % 10) == digit

def mrz_date(date, max_year):
    if not date.isdigit():
        return None
    year = 2000 + int(date[:2])
    return f'{date[4:]}.{date[2:4]}.{year if year <= max_year else year - 100}'

def parse_line2(t):
    if not isinstance(t, str) or len(t) != 44:
        return {}
    number_ok = check(t[0:9], t[9])
    birth_ok = check(t[13:19], t[19])
    extra_ok = check(t[28:42], t[42])
    return {
        'Sex_ru': {'M': 'МУЖ.', 'F': 'ЖЕН.'}.get(t[20]),
        'Birth_date': mrz_date(t[13:19], max_year=2012) if birth_ok else None,
        'Issue_date': mrz_date(t[29:35], max_year=2026) if extra_ok else None,
        'Issue_organisation_code': f'{t[35:38]}-{t[38:41]}' if extra_ok else None,
        'Licence_number': f'{t[0:2]} {t[2]}{t[28]} {t[3:9]}' if number_ok and extra_ok else None,
    }

In [80]:
parse_line2('6122839687RUS1006102M<<<<<<<4250911620004<28')

{'Sex_ru': 'МУЖ.',
 'Birth_date': '10.06.2010',
 'Issue_date': '11.09.2025',
 'Issue_organisation_code': '620-004',
 'Licence_number': '61 24 283968'}

парсим line1

In [81]:
translit = {'А':'A','Б':'B','В':'V','Г':'G','Д':'D','Е':'E','Ё':'E','Ж':'J','З':'Z','И':'I',
    'Й':'Q','К':'K','Л':'L','М':'M','Н':'N','О':'O','П':'P','Р':'R','С':'S','Т':'T','У':'U','Ф':'F',
    'Х':'H','Ц':'C','Ч':'3','Ш':'4','Щ':'W','Ъ':'X','Ы':'Y','Ь':'9','Э':'6','Ю':'7','Я':'8',' ':'<','-':'<'}
inv_translit = {lat: cyr for cyr, lat in translit.items() if cyr not in 'Ё -'}

def to_latin(s):
    return ''.join(translit.get(ch, ch) for ch in s)

def pick_char(a, b, c, cc, low):
    if a == b or (c >= low > cc):
        return a
    if c < low <= cc and a.isalpha() and b.isalpha():
        return b
    return None

def merge(word, conf, cand, cand_conf, low=0.95):
    if len(word) != len(cand):
        return None
    out = [pick_char(a, b, c, cc, low) for a, b, c, cc in zip(word, cand, conf, cand_conf)]
    return None if None in out else ''.join(out)

def mrz_names(line, conf):
    if not isinstance(line, str) or len(line) != 44 or not line.endswith('<'):
        return None
    parts = line[5:].rstrip('<').split('<<', 1)
    last = parts[0]
    rest = parts[1] if len(parts) > 1 else ''
    names = ([last] + rest.split('<') + [''])[:3]
    if not all(c in inv_translit for c in ''.join(names)):
        return None
    starts = [5, 7 + len(names[0]), 8 + len(names[0]) + len(names[1])]
    return [(''.join(inv_translit[c] for c in s), conf[i:i + len(s)]) for s, i in zip(names, starts)]

def yymmdd(date):
    return date[8:10] + date[3:5] + date[:2]

def viz_line2(t, v):
    if not isinstance(t, str):
        return None
    n, c = v['Licence_number'].replace(' ', ''), v['Issue_organisation_code'].replace('-', '')
    return n[:3] + n[4:] + t[9:13] + yymmdd(v['Birth_date']) + t[19:28] + n[3:4] + yymmdd(v['Issue_date']) + c + t[41:]

def mrz_fix(df):
    values = pd.DataFrame({f: df[f'viz.{f}.text'] for f in FIELDS})
    mrz2 = pd.DataFrame(df['mrz.line1'].apply(parse_line2).tolist(), index=df.index)
    viz_lines = [viz_line2(df.loc[doc, 'mrz.line1'], values.loc[doc]) for doc in df.index]
    mrz_viz = pd.DataFrame([parse_line2(t) for t in viz_lines], index=df.index)
    for f in name_fields:
        values[f] = values[f].str.replace('Ё', 'Е')
    names = (values['Last_name_ru'] + '<<' + values['First_name_ru'] + '<' + values['Middle_name_ru']).map(to_latin)
    names_ok = df['mrz.line0'].str[5:] == (names + '<' * 39).str[:39]
    covered = pd.DataFrame({f: names_ok for f in name_fields})
    covered['Sex_ru'] = values['Sex_ru'] == mrz2['Sex_ru']

    for doc in df.index[names_ok == False]:
        mrz = mrz_names(df.loc[doc, 'mrz.line0'], df.loc[doc, 'mrz.chars'][1])
        if mrz is None:
            continue
        merged = []
        for i, f in enumerate(name_fields):
            text, conf = df.loc[doc, f'viz.{f}.chars']
            cand, cand_conf = mrz[i]
            merged.append(merge(text.replace('Ё', 'Е'), conf, cand, cand_conf))
        if None not in merged:
            values.loc[doc, name_fields] = merged
            covered.loc[doc, name_fields] = True

    for f in mrz_fields:
        covered[f] = mrz2[f].notna() | (mrz_viz[f] == values[f])
        values[f] = mrz2[f].fillna(values[f])
    return values, covered

train_values, train_covered = mrz_fix(train_predictions)
test_values, test_covered = mrz_fix(test_predictions)

In [82]:
subdivisions = pd.read_csv('reference/subdivisions.csv')
known = set(' '.join(subdivisions['name']).upper().split())
known_by_len = {}
for v in known:
    if len(v) not in known_by_len:
        known_by_len[len(v)] = []
    known_by_len[len(v)].append(v)

def fix_org(chars):
    text, conf = chars
    words, pos = [], 0
    for w in text.split(' '):
        c = conf[pos:pos + len(w)]
        if w in known:
            cands = [w]
        else:
            cands = [v for v in known_by_len.get(len(w), []) if merge(w, c, v, [1.0] * len(v))]
        words.append(cands[0] if len(cands) == 1 else w)
        pos += len(w) + 1
    fixed = ' '.join(words)
    return fixed if fixed != text and all(w in known for w in words) else None

for df, values in [(train_predictions, train_values), (test_predictions, test_values)]:
    fixed = df['viz.Issue_organization_ru.chars'].map(fix_org)
    values['Issue_organization_ru'] = fixed.fillna(values['Issue_organization_ru'])
    df.loc[fixed.notna(), 'viz.Issue_organization_ru.cmin'] = 1.0
    print('исправлено:', fixed.notna().sum())

исправлено: 83
исправлено: 52


In [83]:
field_ok = pd.DataFrame({f: train_values[f] == train_labels[f'fields.{f}'].map(norm) for f in FIELDS})
y = field_ok.all(axis=1).astype(int)

In [84]:
y.value_counts()

1    1955
0    1045
Name: count, dtype: int64

In [85]:
def is_valid(values, on=pd.Timestamp('2026-10-01')):
    birth = pd.to_datetime(values['Birth_date'], format='%d.%m.%Y', errors='coerce')
    issue = pd.to_datetime(values['Issue_date'], format='%d.%m.%Y', errors='coerce')
    valid = (issue <= on) & (issue >= birth + pd.DateOffset(years=14))
    for age in [20, 45]:
        birthday = birth + pd.DateOffset(years=age)
        valid &= (issue >= birthday) | (birthday > on)
    return valid

In [86]:
noice = ['tr_01665']   # аул Джамбичи существует, в разметке "ДЖАМБИУИ": ошибка разметки, а не OCR. Это очень сильно завышает порог web,
# у неё cmin = 0.9962, у следующей 0.9277
GRID = [0.5, 0.8, 0.9, 0.95, 0.98, 0.99, 0.995, 0.998, 0.999, 0.9995, 0.9999, 0.99999]

cmin = train_predictions[[f'viz.{f}.cmin' for f in two_fields]].min(axis=1)
lost_ok = (train_predictions[[f'viz.{f}.lost' for f in two_fields]] == 0).all(axis=1)
ocr_bad = (field_ok[two_fields].all(axis=1) == False) & (train_predictions.index.isin(noice) == False)
source = train_predictions['source']

def pick(s, margin, rows=True):
    last_error = cmin[rows & lost_ok & ocr_bad & (source == s)].max()
    i = np.searchsorted(GRID, last_error, side='right') + margin
    return GRID[min(i, len(GRID) - 1)]

In [87]:
min_conf = {}
for s in ['web', 'mobile', 'mobile_v2']:
    for margin in range(len(GRID)):
        errors = 0
        for rep in range(20):
            for fit, _ in StratifiedKFold(n_splits=10, shuffle=True, random_state=rep).split(cmin, ocr_bad):
                is_fit = pd.Series(False, index=cmin.index)
                is_fit.iloc[fit] = True
                acc = (is_fit == False) & lost_ok & (source == s) & (cmin >= pick(s, margin, is_fit))
                errors += (acc & ocr_bad).sum()
        print(f'{s}: запас {margin}, порог {pick(s, margin)}, ошибок на отложенных {errors / 20}')
        if errors == 0:
            min_conf[s] = pick(s, margin)
            break
print(min_conf)

web: запас 0, порог 0.95, ошибок на отложенных 0.0
mobile: запас 0, порог 0.95, ошибок на отложенных 0.0
mobile_v2: запас 0, порог 0.999, ошибок на отложенных 0.4
mobile_v2: запас 1, порог 0.9995, ошибок на отложенных 0.0
{'web': 0.95, 'mobile': 0.95, 'mobile_v2': 0.9995}


# accept rule

In [88]:
def accept_rule(df, covered, values):
    ok = covered.all(axis=1) & is_valid(values)
    for f in two_fields:
        ok &= (df[f'viz.{f}.cmin'] >= df['source'].map(min_conf)) & (df[f'viz.{f}.lost'] == 0)
    return ok

train_accept = accept_rule(train_predictions, train_covered, train_values)
test_accept = accept_rule(test_predictions, test_covered, test_values)
n, k = train_accept.sum(), (train_accept & (y == 0)).sum()
print('train: принято', n, 'с ошибкой:', k)#вроде как эти ошибки - шум, проверял внизу
print('test: принято', test_accept.sum())

train: принято 1550 с ошибкой: 10
test: принято 1036


In [89]:
train_predictions.dtypes

source                                str
quality.sharpness                 float64
quality.doc_px                      int64
quality.glare                     float64
viz.Last_name_ru.box               object
                                   ...   
viz.Issue_organization_ru.cmin    float64
viz.Issue_organization_ru.lost    float64
mrz.chars                          object
mrz.line0                             str
mrz.line1                             str
Length: 102, dtype: object

# Бустинг

In [90]:
def make_features(df, covered):
    x = df.select_dtypes('number').dropna(axis=1, how='all')
    x['n_covered'] = covered.sum(axis=1)
    x['mobile_v2'] = df['source'] == 'mobile_v2'
    return x

x = make_features(train_predictions, train_covered)

In [91]:
model = lgb.LGBMClassifier(max_depth=4, n_estimators=500, learning_rate=0.01, random_state=42, verbose=-1)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

p = np.zeros(len(y))
for train_index, val_index in cv.split(x, y):
    model.fit(x.iloc[train_index], y.iloc[train_index],
          eval_X=x.iloc[val_index], eval_y=y.iloc[val_index],
          eval_metric='auc', callbacks=[lgb.log_evaluation(period=100)])

    p[val_index] = model.predict_proba(x.iloc[val_index])[:, 1]
    print('auc', round(roc_auc_score(y.iloc[val_index], p[val_index]), 3))

s = pd.Series(p, index=y.index)


[100]	valid_0's auc: 0.982437	valid_0's binary_logloss: 0.295215
[200]	valid_0's auc: 0.98592	valid_0's binary_logloss: 0.200587
[300]	valid_0's auc: 0.986583	valid_0's binary_logloss: 0.16696
[400]	valid_0's auc: 0.989919	valid_0's binary_logloss: 0.153502
[500]	valid_0's auc: 0.985724	valid_0's binary_logloss: 0.145984
auc 0.986
[100]	valid_0's auc: 0.949348	valid_0's binary_logloss: 0.325878
[200]	valid_0's auc: 0.96821	valid_0's binary_logloss: 0.240155
[300]	valid_0's auc: 0.969976	valid_0's binary_logloss: 0.21009
[400]	valid_0's auc: 0.973558	valid_0's binary_logloss: 0.193593
[500]	valid_0's auc: 0.97422	valid_0's binary_logloss: 0.183152
auc 0.974
[100]	valid_0's auc: 0.948489	valid_0's binary_logloss: 0.30591
[200]	valid_0's auc: 0.984596	valid_0's binary_logloss: 0.203693
[300]	valid_0's auc: 0.98592	valid_0's binary_logloss: 0.16809
[400]	valid_0's auc: 0.986607	valid_0's binary_logloss: 0.152274
[500]	valid_0's auc: 0.986509	valid_0's binary_logloss: 0.146075
auc 0.987
[10

# Оценка бустинга

In [92]:
print(roc_auc_score(y, s))

0.9842976541563161


In [113]:
err_p = s[y == 0].sort_values(ascending=False).values
for max_bad in [0, 1,2,3,4, 5, 10]:
    print(f'ошибок <= {max_bad}: принято {(s > err_p[max_bad]).sum()}')

ошибок <= 0: принято 222
ошибок <= 1: принято 319
ошибок <= 2: принято 325
ошибок <= 3: принято 429
ошибок <= 4: принято 694
ошибок <= 5: принято 1313
ошибок <= 10: принято 1620


# Проверка шума в разметке train_predictions

In [126]:
for d in s[y == 0].sort_values(ascending=False).head(5).index:
    wrong = field_ok.columns[field_ok.loc[d] == False]
    print(d, train_values.loc[d, wrong].tolist(), train_labels.loc[d, ['fields.' + f for f in wrong]].tolist())

tr_02183 ['КИТУРКО'] ['КИТУРКС']
tr_01651 ['ЗИЯНУРОВ'] ['ЗИЯНУРСВ']
tr_02818 ['ЕВГЕНИЙ'] ['ЕВТЕНИЙ']
tr_01185 ['МУЗЫЧУК'] ['МУЗЫУУК']
tr_01765 ['ХАСАНЗЯН'] ['ХАОАНЗЯН']


In [125]:
names = (train_values['Last_name_ru'] + '<<' + train_values['First_name_ru'] + '<' + train_values['Middle_name_ru']).map(to_latin)
same = train_predictions['mrz.line0'].str[5:].str.rstrip('<') == names
noise = same & (field_ok[name_fields].all(axis=1) == False)

for d in train_predictions.index[train_accept & (y == 0)]:
    f = field_ok.columns[field_ok.loc[d] == False][0]
    print(d, train_values.loc[d, f],':', train_labels.loc[d, 'fields.' + f])

tr_00369 ФАРИДА : ФАРИЛА
tr_00747 БАЛТРУШЕЕВИЧ : ЬАЛТРУШЕЕВИЧ
tr_01185 МУЗЫЧУК : МУЗЫУУК
tr_01506 КАБЛЯХБАР : КАБЛЯЖБАР
tr_01651 ЗИЯНУРОВ : ЗИЯНУРСВ
tr_01665 АУЛ ДЖАМБИЧИ РЕСП. АДЫГЕЯ : АУЛ ДЖАМБИУИ РЕСП. АДЫГЕЯ
tr_01765 ХАСАНЗЯН : ХАОАНЗЯН
tr_02183 КИТУРКО : КИТУРКС
tr_02287 МУТЯНСКИЙ : МУТЯНСЖИЙ
tr_02818 ЕВГЕНИЙ : ЕВТЕНИЙ


Например, в паспорте ЕВГЕНИЙ, в MRZ EVGENIQ, а в разметке ЕВТЕНИЙ. При этом уверенность Г = 1.0, mrz и фио также совпадают

In [102]:
text, conf = train_predictions.loc['tr_02818', 'viz.First_name_ru.chars']
list(zip(text, conf))

[('Е', 0.9984000000000001),
 ('В', 0.9991000000000001),
 ('Г', 1.0),
 ('Е', 0.9994000000000001),
 ('Н', 0.9996),
 ('И', 0.9999),
 ('Й', 0.9999)]

In [103]:
text, conf = train_predictions.loc['tr_01765', 'viz.First_name_ru.chars']
list(zip(text, conf))

[('Х', 0.9996),
 ('А', 0.9994000000000001),
 ('С', 0.9999),
 ('А', 0.9999),
 ('Н', 0.9995),
 ('З', 0.9997),
 ('Я', 0.9989),
 ('Н', 0.9996)]